In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from early_stopping import EarlyStopping

# Giả định một mạng neural đơn giản
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)

# Tạo dữ liệu giả lập
x_train = torch.rand(100, 10)
y_train = torch.rand(100, 1)
x_val = torch.rand(20, 10)
y_val = torch.rand(20, 1)

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Khởi tạo model, loss function, optimizer
model = SimpleModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Khởi tạo EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    verbose=1,
    restore_best_weights=True,
    save_path='./best_model.pt'
)

# Khởi tạo ModelCheckpoint
model_checkpoint = ModelCheckpoint(
    filepath='./checkpoint_model.pth',  # Sử dụng tên cố định để ghi đè lên file
    monitor='val_loss',
    verbose=1,
    save_best_only=True
)

# Vòng lặp huấn luyện
num_epochs = 50
for epoch in range(num_epochs):
    # Huấn luyện
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(x_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Đánh giá trên tập validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            predictions = model(x_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    # In log
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # Lưu checkpoint
    model_checkpoint(val_loss, model, epoch)

    # Kiểm tra điều kiện EarlyStopping
    early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

# Model tốt nhất đã được lưu vào ./best_model.pt
print("Training complete. Best weights restored (if enabled).")


Epoch 0: train_loss=0.7860, val_loss=0.5015
Saving model to ./checkpoint_model.pth at epoch 0.
Saved best model weights at epoch 0 to './best_model.pt'
Epoch 1: train_loss=0.7433, val_loss=0.4522
Saving model to ./checkpoint_model.pth at epoch 1.
Saved best model weights at epoch 1 to './best_model.pt'
Epoch 2: train_loss=0.6024, val_loss=0.4067
Saving model to ./checkpoint_model.pth at epoch 2.
Saved best model weights at epoch 2 to './best_model.pt'
Epoch 3: train_loss=0.5540, val_loss=0.3660
Saving model to ./checkpoint_model.pth at epoch 3.
Saved best model weights at epoch 3 to './best_model.pt'
Epoch 4: train_loss=0.5008, val_loss=0.3293
Saving model to ./checkpoint_model.pth at epoch 4.
Saved best model weights at epoch 4 to './best_model.pt'
Epoch 5: train_loss=0.4693, val_loss=0.2962
Saving model to ./checkpoint_model.pth at epoch 5.
Saved best model weights at epoch 5 to './best_model.pt'
Epoch 6: train_loss=0.3956, val_loss=0.2662
Saving model to ./checkpoint_model.pth at ep

# Test ReduceLROnPlateau

In [6]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Giả sử bạn có một mạng đơn giản
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)

# Dữ liệu giả lập
x_train = torch.rand(100, 10)
y_train = torch.rand(100, 1)
x_val = torch.rand(20, 10)
y_val = torch.rand(20, 1)

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Khởi tạo model, optimizer, loss function
model = SimpleModel()
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Tạo scheduler giảm learning rate khi val_loss không cải thiện
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

# Huấn luyện model
num_epochs = 50
for epoch in range(num_epochs):
    # Huấn luyện
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(x_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Đánh giá trên tập validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            predictions = model(x_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    # In log
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # Gọi scheduler để điều chỉnh learning rate
    scheduler.step(val_loss)

    print(f"Learning rate: {optimizer.param_groups[0]['lr']}")


Epoch 0: train_loss=0.3948, val_loss=0.1254
Learning rate: 0.01
Epoch 1: train_loss=0.1387, val_loss=0.0638
Learning rate: 0.01
Epoch 2: train_loss=0.1381, val_loss=0.1076
Learning rate: 0.01
Epoch 3: train_loss=0.1154, val_loss=0.0838
Learning rate: 0.01
Epoch 4: train_loss=0.1015, val_loss=0.0610
Learning rate: 0.01
Epoch 5: train_loss=0.0914, val_loss=0.0540
Learning rate: 0.01
Epoch 6: train_loss=0.0933, val_loss=0.0528
Learning rate: 0.01
Epoch 7: train_loss=0.0867, val_loss=0.0535
Learning rate: 0.01
Epoch 8: train_loss=0.0865, val_loss=0.0582
Learning rate: 0.01
Epoch 9: train_loss=0.0840, val_loss=0.0594
Learning rate: 0.01
Epoch 10: train_loss=0.0832, val_loss=0.0550
Learning rate: 0.01
Epoch 11: train_loss=0.0842, val_loss=0.0505
Learning rate: 0.01
Epoch 12: train_loss=0.0812, val_loss=0.0510
Learning rate: 0.01
Epoch 13: train_loss=0.0897, val_loss=0.0546
Learning rate: 0.01
Epoch 14: train_loss=0.0780, val_loss=0.0536
Learning rate: 0.01
Epoch 15: train_loss=0.0809, val_lo

/home/vohoang/miniconda3/envs/torch/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
